In [ ]:
from google.colab import drive
drive.mount('/content/drive')
exec(open("/content/drive/MyDrive/imdb_peft_project/code/lora-imdb-classifier/00_colab_setup.py").read())

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ /content/drive/MyDrive/imdb_peft_project
✓ /content/drive/MyDrive/imdb_peft_project/checkpoints/roberta_lora
✓ /content/drive/MyDrive/imdb_peft_project/checkpoints/deberta_lora
✓ /content/drive/MyDrive/imdb_peft_project/oof_predictions
✓ /content/drive/MyDrive/imdb_peft_project/results
✓ /content/drive/MyDrive/imdb_peft_project/notebooks
✓ /content/drive/MyDrive/imdb_peft_project/code

Folder structure ready.
Enter GitHub Token: ··········
Repository exists, pulling latest changes...
✓ Pull complete.

Repository path: /content/drive/MyDrive/imdb_peft_project/code/lora-imdb-classifier
SETUP COMPLETE
Drive folder  : /content/drive/MyDrive/imdb_peft_project
GitHub repo   : /content/drive/MyDrive/imdb_peft_project/code/lora-imdb-classifier

Available functions:
  push_to_github('message')        → push code to GitHub
  save_code_to_rep

In [ ]:
!pip install transformers peft accelerate torchao scikit-learn xgboost safetensors --upgrade -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 139.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 120.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 152.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 27.2 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
import os
import safetensors.torch as st
from transformers import (RobertaTokenizer, RobertaForSequenceClassification,
                          DebertaV2Tokenizer, DebertaV2ForSequenceClassification,
                          Trainer, TrainingArguments)
from peft import PeftModel, LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_score, recall_score,
                             roc_auc_score, confusion_matrix,
                             roc_curve, auc)
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [ ]:
# Define paths
DIRS["oof_v2"]            = f"{DIRS['root']}/oof_predictions_v2"
DIRS["oof_v2_best"]       = f"{DIRS['root']}/oof_predictions_v2_best"
DIRS["checkpoints_v2"]    = f"{DIRS['root']}/checkpoints/roberta_lora_v2"
DIRS["checkpoints2_best"] = f"{DIRS['root']}/checkpoints/deberta_lora_v2_best"

# Load data
train_df = pd.read_parquet(f"{DIRS['root']}/train_df_v2.parquet")
test_df  = pd.read_parquet(f"{DIRS['root']}/test_df_v2.parquet")
y_test   = test_df["label"].values

print(f"Train : {len(train_df)} samples")
print(f"Test  : {len(test_df)} samples")

Train : 25000 samples
Test  : 25000 samples


In [ ]:
def head_tail_truncate_v2(text, tokenizer, max_len=512, head_len=256):
    """V2: Equal split — first 256 + last 256 tokens."""
    tail_len = max_len - head_len
    tokens = tokenizer(text, add_special_tokens=False,
                       truncation=False, return_tensors=None)
    input_ids      = tokens["input_ids"]
    attention_mask = tokens["attention_mask"]
    if len(input_ids) > max_len - 2:
        input_ids      = input_ids[:head_len] + input_ids[-tail_len:]
        attention_mask = attention_mask[:head_len] + attention_mask[-tail_len:]
    return tokenizer(
        tokenizer.decode(input_ids),
        max_length=max_len,
        padding="max_length",
        truncation=True,
        return_tensors=None
    )

class IMDBDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512, head_len=256):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.head_len  = head_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text  = self.df.loc[idx, "text_clean"]
        label = self.df.loc[idx, "label"]
        encoding = head_tail_truncate_v2(
            text, self.tokenizer, self.max_len, self.head_len
        )
        return {
            "input_ids":      torch.tensor(encoding["input_ids"],      dtype=torch.long),
            "attention_mask": torch.tensor(encoding["attention_mask"], dtype=torch.long),
            "labels":         torch.tensor(label,                      dtype=torch.long),
        }

print("✓ Dataset class ready.")

✓ Dataset class ready.


In [ ]:
print("Loading RoBERTa V2...")
roberta_tokenizer    = RobertaTokenizer.from_pretrained("roberta-base")
roberta_test_dataset = IMDBDataset(test_df, roberta_tokenizer)

base_model    = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)
roberta_model = PeftModel.from_pretrained(base_model, f"{DIRS['checkpoints_v2']}/final")
roberta_model.eval()
print("✓ RoBERTa V2 loaded.")

training_args = TrainingArguments(
    output_dir="/tmp/eval_r", per_device_eval_batch_size=64, report_to="none"
)
trainer = Trainer(model=roberta_model, args=training_args)
preds_output       = trainer.predict(roberta_test_dataset)
roberta_test_probs = torch.softmax(
    torch.tensor(preds_output.predictions), dim=-1
)[:, 1].numpy()

print(f"✓ RoBERTa test predictions ready. Shape: {roberta_test_probs.shape}")
print(f"  Sample probs: {roberta_test_probs[:5]}")

del roberta_model, trainer
torch.cuda.empty_cache()

Loading RoBERTa V2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✓ RoBERTa V2 loaded.


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (544 > 512). Running this sequence through the model will result in indexing errors
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


✓ RoBERTa test predictions ready. Shape: (25000,)
  Sample probs: [0.00117237 0.3785614  0.00233001 0.00169863 0.9975853 ]


In [ ]:
print("Loading DeBERTa V2 Best...")
deberta_tokenizer    = DebertaV2Tokenizer.from_pretrained("microsoft/deberta-v3-base")
deberta_test_dataset = IMDBDataset(test_df, deberta_tokenizer)

# Load base model
base_model = DebertaV2ForSequenceClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=2,
    torch_dtype=torch.float32,
    ignore_mismatched_sizes=True
)

# Add LoRA r=32
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=32, lora_alpha=64, lora_dropout=0.1,
    target_modules=["query_proj", "value_proj"],
    bias="none"
)
deberta_model = get_peft_model(base_model, lora_config)

# Load weights
epoch3_path = f"{DIRS['checkpoints2_best']}/epoch_3"
weights     = st.load_file(f"{epoch3_path}/adapter_model.safetensors")
extra       = torch.load(f"{epoch3_path}/extra_weights.pt", map_location="cpu")

new_weights = {}
for k, v in weights.items():
    if "lora_A.weight" in k:
        new_weights[k.replace("lora_A.weight", "lora_A.default.weight")] = v
    elif "lora_B.weight" in k:
        new_weights[k.replace("lora_B.weight", "lora_B.default.weight")] = v

new_weights["base_model.model.classifier.modules_to_save.default.weight"] = extra["classifier.weight"]
new_weights["base_model.model.classifier.modules_to_save.default.bias"]   = extra["classifier.bias"]
new_weights["base_model.model.classifier.original_module.weight"]          = extra["classifier.weight"]
new_weights["base_model.model.classifier.original_module.bias"]            = extra["classifier.bias"]
new_weights["base_model.model.pooler.dense.weight"]                        = extra["pooler.weight"]
new_weights["base_model.model.pooler.dense.bias"]                          = extra["pooler.bias"]

missing, unexpected = deberta_model.load_state_dict(new_weights, strict=False)
deberta_model = deberta_model.to(torch.float32).to(device)
deberta_model.eval()
print(f"Missing: {len(missing)} | Unexpected: {len(unexpected)}")

# Quick test
sample = deberta_test_dataset[0]
with torch.no_grad():
    out   = deberta_model(
        input_ids      = sample["input_ids"].unsqueeze(0).to(device),
        attention_mask = sample["attention_mask"].unsqueeze(0).to(device)
    )
    probs = torch.softmax(out.logits, dim=-1)
print(f"Quick test — Label: {sample['labels'].item()} | Probs: {probs[0].tolist()}")

Loading DeBERTa V2 Best...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.den

Missing: 198 | Unexpected: 0
Quick test — Label: 0 | Probs: [0.9977108240127563, 0.002289216499775648]


In [ ]:
training_args = TrainingArguments(
    output_dir="/tmp/eval_d",
    per_device_eval_batch_size=32,
    report_to="none"
)
trainer = Trainer(model=deberta_model, args=training_args)
preds_output        = trainer.predict(deberta_test_dataset)
deberta_test_probs  = torch.softmax(
    torch.tensor(preds_output.predictions), dim=-1
)[:, 1].numpy()

print(f"✓ DeBERTa Best test predictions ready. Shape: {deberta_test_probs.shape}")
print(f"  Sample probs: {deberta_test_probs[:5]}")

del deberta_model, trainer
torch.cuda.empty_cache()

✓ DeBERTa Best test predictions ready. Shape: (25000,)
  Sample probs: [0.00228922 0.02945386 0.0023262  0.00151101 0.9975981 ]


In [ ]:
# Load OOF predictions
roberta_oof = np.load(f"{DIRS['oof_v2']}/roberta_v2_fold4.npy")
deberta_oof = np.load(f"{DIRS['oof_v2_best']}/deberta_v2_best_fold4.npy")

print(f"RoBERTa OOF shape: {roberta_oof.shape}")
print(f"DeBERTa OOF shape: {deberta_oof.shape}")

y_train = train_df["label"].values

# Meta feature matrices
X_meta_train = np.column_stack([roberta_oof, deberta_oof])
X_meta_test  = np.column_stack([roberta_test_probs, deberta_test_probs])

print(f"Meta train shape: {X_meta_train.shape}")
print(f"Meta test shape : {X_meta_test.shape}")

RoBERTa OOF shape: (25000,)
DeBERTa OOF shape: (25000,)
Meta train shape: (25000, 2)
Meta test shape : (25000, 2)


In [ ]:
def evaluate_meta(name, y_true, y_pred, y_prob):
    return {
        "model"    : name,
        "accuracy" : round(accuracy_score(y_true, y_pred), 4),
        "f1"       : round(f1_score(y_true, y_pred), 4),
        "precision": round(precision_score(y_true, y_pred), 4),
        "recall"   : round(recall_score(y_true, y_pred), 4),
        "roc_auc"  : round(roc_auc_score(y_true, y_prob), 4),
    }

results = []

# ── 1. Weighted Average ───────────────────────────────────────
wa_probs = roberta_test_probs * 0.5 + deberta_test_probs * 0.5
wa_preds = (wa_probs > 0.5).astype(int)
results.append(evaluate_meta("Weighted Average", y_test, wa_preds, wa_probs))
print("✓ Weighted Average complete.")

# ── 2. Logistic Regression ────────────────────────────────────
lr = LogisticRegression(random_state=SEED, max_iter=1000)
lr.fit(X_meta_train, y_train)
lr_probs = lr.predict_proba(X_meta_test)[:, 1]
lr_preds = lr.predict(X_meta_test)
res = evaluate_meta("Logistic Regression", y_test, lr_preds, lr_probs)
res["roberta_weight"] = round(lr.coef_[0][0], 4)
res["deberta_weight"] = round(lr.coef_[0][1], 4)
results.append(res)
print(f"✓ LR — RoBERTa w: {lr.coef_[0][0]:.4f}, DeBERTa w: {lr.coef_[0][1]:.4f}")

# ── 3. MLP ────────────────────────────────────────────────────
mlp = MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu",
                    max_iter=500, random_state=SEED)
mlp.fit(X_meta_train, y_train)
mlp_probs = mlp.predict_proba(X_meta_test)[:, 1]
mlp_preds = mlp.predict(X_meta_test)
results.append(evaluate_meta("MLP", y_test, mlp_preds, mlp_probs))
print("✓ MLP complete.")

# ── 4. GradientBoosting ───────────────────────────────────────
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                max_depth=3, random_state=SEED)
gb.fit(X_meta_train, y_train)
gb_probs = gb.predict_proba(X_meta_test)[:, 1]
gb_preds = gb.predict(X_meta_test)
results.append(evaluate_meta("GradientBoosting", y_test, gb_preds, gb_probs))
print("✓ GradientBoosting complete.")

# ── 5. XGBoost ────────────────────────────────────────────────
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3,
                    random_state=SEED, eval_metric="logloss", verbosity=0)
xgb.fit(X_meta_train, y_train)
xgb_probs = xgb.predict_proba(X_meta_test)[:, 1]
xgb_preds = xgb.predict(X_meta_test)
results.append(evaluate_meta("XGBoost", y_test, xgb_preds, xgb_probs))
print("✓ XGBoost complete.")

# ── Results table ─────────────────────────────────────────────
results_df = pd.DataFrame(results)
print(f"\n{'='*75}")
print("FINAL META-LEARNER RESULTS (RoBERTa V2 + DeBERTa Best)")
print(f"{'='*75}")
print(results_df[["model", "accuracy", "f1", "precision",
                   "recall", "roc_auc"]].to_string(index=False))

✓ Weighted Average complete.
✓ LR — RoBERTa w: 3.4965, DeBERTa w: 3.7139
✓ MLP complete.
✓ GradientBoosting complete.
✓ XGBoost complete.

FINAL META-LEARNER RESULTS (RoBERTa V2 + DeBERTa Best)
              model  accuracy     f1  precision  recall  roc_auc
   Weighted Average    0.9606 0.9608     0.9553  0.9663   0.9918
Logistic Regression    0.9611 0.9613     0.9575  0.9651   0.9918
                MLP    0.9612 0.9614     0.9575  0.9652   0.9918
   GradientBoosting    0.9584 0.9586     0.9532  0.9640   0.9901
            XGBoost    0.9606 0.9608     0.9566  0.9650   0.9914


In [ ]:
save_results({"meta_learner_best_final": results},
             "meta_learner_best_final_results.json")

NOTEBOOK_NAME = "06d_meta_learner_best"
!jupyter nbconvert --to script \
  "/content/drive/MyDrive/Colab Notebooks/{NOTEBOOK_NAME}.ipynb" \
  --output-dir "/content/"
import os
os.rename(f"/content/{NOTEBOOK_NAME}.txt",
          f"/content/{NOTEBOOK_NAME}.py")
save_code_to_repo(f"/content/{NOTEBOOK_NAME}.py")
push_to_github("final meta learner best MLP 0.9612")

✓ Results saved: /content/drive/MyDrive/imdb_peft_project/results/meta_learner_best_final_results.json
[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/06d_meta_learner_best.ipynb to script
[NbConvertApp] Writing 7790 bytes to /content/06d_meta_learner_best.txt
✓ 06d_meta_learner_best.py → copied to repository.
✓ Pushed to GitHub: 'final meta learner best MLP 0.9612'
